<a href="https://colab.research.google.com/github/Hejran-M/Adversarial-defense-for-battery-state-of-health-prediction-models/blob/main/fgsm_attack_and_adverserial_training20240218.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Impoprt Libraries**

inmport libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv

import sys
import time

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn import metrics

import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D  # Updated import statement
from tensorflow.keras.utils import plot_model
from keras import backend as K
from keras.models import load_model

Import the necessary module to work with Google Drive in Colab.
Mount Google Drive to a specific directory (/content/drive), allowing you to access, read, and write files between Google Drive and your Colab notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

This line reads a CSV file named T25 (1).csv from your Google Drive's "MyDrive" folder.
The file's contents are loaded into a pandas DataFrame, stored in the variable df.
You can now use df to perform data analysis, manipulation, and visualization tasks within your Colab notebook.
Example Usage

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/T25 (1).csv')

The line of code retrieves all the values from the 'Dataset' column of the DataFrame df, converts them into a list, and then creates a set from this list.
The result is a set containing only the unique values from the 'Dataset' column of the DataFrame.

In [ ]:
set(df['Dataset'].tolist())

In [ ]:
df.columns

Line 1: X = [eval(i) for i in df['resized_scaled'].tolist()]
df['resized_scaled']: This extracts the column named 'resized_scaled' from the pandas DataFrame df. The result is a pandas Series object containing the values of the 'resized_scaled' column.

.tolist(): This method converts the pandas Series into a Python list, which contains all the elements from the 'resized_scaled' column.

[eval(i) for i in df['resized_scaled'].tolist()]: This is a list comprehension that iterates over each element i in the list created from the 'resized_scaled' column. The function eval(i) evaluates the string i as a Python expression and returns the result.

eval(): This function parses the string passed to it and executes it as a Python expression. This is often used when the elements of the list are strings representing complex data structures, like lists or arrays in string format, that need to be converted back to their original form.
X = [...]: The result of the list comprehension is assigned to the variable X. At this point, X is a list containing the evaluated results from the 'resized_scaled' column.

Line 2: X = np.array(X)
np.array(X): This line converts the list X into a NumPy array. NumPy arrays are a more efficient data structure than Python lists for numerical data and are widely used in scientific computing for their speed and functionality.

X = np.array(X): The NumPy array created from X replaces the original list X. This means that after this line, X is now a NumPy array.

Line 3: y = np.array(df['SOH'].tolist())
df['SOH']: This extracts the 'SOH' column from the DataFrame df, resulting in a pandas Series object containing the values of the 'SOH' column.

.tolist(): This method converts the pandas Series into a Python list, which contains all the elements from the 'SOH' column.

np.array(df['SOH'].tolist()): This converts the list of values from the 'SOH' column into a NumPy array.

y = ...: The resulting NumPy array is assigned to the variable y.

Summary of What the Code Does
Converts the 'resized_scaled' column from the DataFrame df (which likely contains data in string format that needs to be evaluated) into a list of evaluated expressions, then into a NumPy array X.

Converts the 'SOH' column from the DataFrame df into a NumPy array y.

In [ ]:
X = [eval(i) for i in df['resized_scaled'].tolist()]
X = np.array(X)
y = np.array(df['SOH'].tolist()) ##%changed

Purpose of this part of the Code
Stratified Sampling:

The code is preparing labels for stratified random sampling. This is a sampling technique used to ensure that different categories (or strata) are proportionately represented in the sample.
By converting SOH values into numerical labels representing different ranges, the code ensures that samples taken will have a balanced representation of each range. This is especially useful in machine learning to avoid biased training data.
Discretizing Continuous Data:

The SOH values, which are continuous, are being discretized into bins or ranges. This is a common preprocessing step in machine learning to simplify the data and manage it more effectively.
Result:

At the end of this process, label_l contains numerical labels (from 0 to 16) representing the different SOH ranges. This can now be used for stratified sampling or other analyses where categorized data is needed.

In [ ]:
# labeling for stratified random sampling
label = df['SOH'].tolist()
label_l = []
for cnt in range(len(label)):
    i = label[cnt]
    if i>=1.0 :
        lab = '1~'
        num = 0
    elif 0.98<=i<1.0 :
        lab = '0.98~1'
        num = 1
    elif 0.96<=i<0.98 :
        lab = '0.96~0.98'
        num = 2
    elif 0.94<=i<0.96 :
        lab = '0.94~0.96'
        num = 3
    elif 0.92<=i<0.94 :
        lab = '0.92~0.94'
        num = 4
    elif 0.90<=i<0.92 :
        lab = '0.90~0.92'
        num = 5

    elif 0.88<=i<0.9 :
        lab = '0.88~0.9'
        num = 6
    elif 0.86<=i<0.88 :
        lab = '0.86~0.88'
        num = 7
    elif 0.84<=i<0.86 :
        lab = '0.84~0.86'
        num = 8
    elif 0.82<=i<0.84 :
        lab = '0.82~0.84'
        num = 9
    elif 0.80<=i<0.82 :
        lab = '0.80~0.82'
        num = 10


    elif 0.78<=i<0.8 :
        lab = '0.78~0.8'
        num = 11
    elif 0.76<=i<0.78 :
        lab = '0.76~0.78'
        num = 12
    elif 0.74<=i<0.76 :
        lab = '0.74~0.76'
        num = 13
    elif 0.72<=i<0.74 :
        lab = '0.72~0.74'
        num = 14
    elif 0.70<=i<0.72 :
        lab = '0.70~0.72'
        num = 15

    else:
        lab = '~0.7'
        num = 16
    label_l.append(num)

StratifiedShuffleSplit is used to create stratified train-test splits.
10 Splits are generated, each with a 70-30 training-to-testing ratio.
The loop iterates through these splits, extracting and preparing the training and testing datasets and their corresponding labels. This prepares the data for robust evaluation and training of machine learning models, ensuring that class distributions are consistent across splits.








In [ ]:
sss = StratifiedShuffleSplit(n_splits = 10, test_size = 0.3)
for train_idx, test_idx in sss.split(X, np.array(label_l)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = np.array(y)[train_idx], np.array(y)[test_idx]
    y_valid = np.array(label_l)[test_idx]

Stratified Resampling: This approach uses stratified resampling to split the already defined test set into a validation and a final test set.
Split Sizes: The second split divides 70% of the original X_test data into X_valid and 30% into X_test_f.
Class Distribution: Stratification ensures that the distribution of class labels in both sets reflects the original dataset distribution, which is useful for model evaluation to prevent biases due to class imbalance.

In [ ]:
sss = StratifiedShuffleSplit(n_splits = 10, test_size = 0.3)
for train_idx, test_idx1 in sss.split(X_test, y_valid):
    X_valid, X_test_f = X_test[train_idx], X_test[test_idx1]
    y_valid, y_test_f = np.array(y_test)[train_idx], np.array(y_test)[test_idx1]
X_valid.shape

Purpose: This code is filtering through a specific subset of the DataFrame df, looking at the 'Dataset' column for specific substrings ('LFP', 'NCA', 'NMC'). When these substrings are found, the corresponding string is added to the data_list.
Output: After running the loop, data_list will contain a list of strings ('LFP', 'NCA', 'NMC') corresponding to the entries found in the specified subset of the 'Dataset' column.
Application: This might be used to count or analyze the types of datasets represented in the final test set, or for further processing where knowing the type of dataset (e.g., battery chemistries LFP, NCA, NMC) is necessary.

In [ ]:
data_list = []
for i in df['Dataset'][test_idx[test_idx1]].tolist():
    if 'LFP' in i:
        data_list.append('LFP')
    if 'NCA' in i:
        data_list.append('NCA')
    if 'NMC' in i:
        data_list.append('NMC')

Purpose of Reassignments: These lines are renaming or aliasing the variables X_train, X_valid, and X_test_f to x_train, x_valid, and x_test, respectively. This might be done for clarity, convenience, or to fit a specific naming convention used later in the code.
Usage: After these assignments, x_train, x_valid, and x_test will be used in further steps such as model training, validation, and testing respectively.

In [ ]:
x_train = X_train
x_valid = X_valid
x_test = X_test_f

Data Preparation: The code snippet prepares image data (or similarly structured data) for input into a convolutional neural network (CNN). This includes reshaping the data into the required format and normalizing pixel values.
Reshape: Ensures that the data is in the correct 4D format for CNNs.
Normalization: Scales pixel values to improve the training process.
Batch Size: Defines the size of batches for training.

In [ ]:
rows = len(X[0])
cols = 3
input_shape = (rows, cols, 1)
x_train = x_train.reshape(x_train.shape[0], rows, cols, 1)
x_test = x_test.reshape(x_test.shape[0], rows, cols, 1)
x_valid = x_valid.reshape(x_valid.shape[0], rows, cols, 1)

x_train = x_train.astype('float32') /255.0
x_test = x_test.astype('float32')/255.0
x_valid = x_valid.astype('float32')/255.0

batch_size = 64
x_train.shape

Sequential Model: This setup uses a simple Sequential model where each layer is added one after the other.
Convolutional Layers: Extract features from the input data using convolutional operations.
Pooling Layer: Reduces the spatial dimensions of the feature maps to make the model less sensitive to small translations in the input.
Flatten Layer: Converts the 2D data into 1D to connect to fully connected layers.
Dense Layers: Perform classification or regression tasks based on the features extracted by convolutional layers.
Model Summary: Provides a detailed overview of the model architecture and parameters.








In [ ]:
###
model = Sequential()
model.add(Conv2D(32, kernel_size = (2,2), strides = (1,1), padding = 'same', activation = 'relu', input_shape = input_shape))
model.add(Conv2D(64, kernel_size = (2,2), strides = (1,1), padding = 'same', activation = 'relu', input_shape = input_shape))
model.add(MaxPooling2D(pool_size = (2,2), strides = (1,1)))
model.add(Flatten())
model.add(Dense(64,activation = 'relu'))
model.add(Dense(1))
model.summary()
###

Timing: Records and prints the start time to measure training duration.
Compilation: Prepares the model with a loss function, optimizer, and evaluation metrics.
Training: Fits the model to the training data and evaluates it on the validation data for a specified number of epochs. The training progress is displayed with detailed logs due to verbose=1.


In [ ]:
start = time.time()  # Record start time
epochs = 40
print('Start time:', start)

model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mse'])

hist2 = model.fit(x_train, y_train, epochs=epochs, verbose=1, validation_data=(x_valid, y_valid))

end = time.time()  # Record end time
training_duration = end - start  # Compute duration

print(f'Training completed in {training_duration:.2f} seconds ({training_duration/60:.2f} minutes).')

In [ ]:
start = time.time()
epochs =40
print('start time : ' , start)
model.compile(loss = 'mean_squared_error', optimizer = 'adam', metrics = ['mse'])
hist2 = model.fit(x_train, y_train, epochs = epochs, verbose = 1, validation_data = (x_valid, y_valid))

Imports: Ensure you have matplotlib.pyplot and numpy imported for plotting and handling data.
Data Extraction: Fetches loss values from the history object after training.
Epoch Indices: Creates a sequence of integers to use as x-axis values.
Plotting: Plots training and validation loss curves with different colors and markers for visual distinction.
Legend and Title: Adds a legend and title to the plot for clarity.
Display Plot: Adds labels and a grid to the plot, then displays it

In [ ]:
y_vloss = hist2.history['val_loss']
y_loss = hist2.history['loss']
x_len = np.arange(len(y_loss))

plt.plot(x_len, y_vloss, marker='.', c='red', label="Validation-set Loss")
plt.plot(x_len, y_loss, marker='.', c='blue', label="Train-set Loss")
plt.legend()
plt.title('epoch40_loss')

model.predict is used to generate predictions from the trained model, and setting verbose=0 ensures the process runs quietly without progress updates.

In [ ]:
predict = model.predict(x_test, verbose = 0)

p = []:

Purpose: Initializes an empty list p to store the predicted values.
Description: This list will hold the predicted State of Health (SOH) values from the model.
t = []:

Purpose: Initializes an empty list t to store the actual test values.
Description: This list will hold the true SOH values from the test set for comparison.
for i in range(len(predict))::

Purpose: Iterates over the range of indices from 0 to the length of predict.
Description: This loop will process each element in the predict array and corresponding elements in y_test_f.
p.append(predict[i]):

Purpose: Adds the predicted value for index i to the list p.
Description: This saves each prediction in the list p.
t.append(y_test_f[i]):

Purpose: Adds the actual value for index i from y_test_f to the list t.
Description: This saves each actual test value in the list t.
#print('predicted SOH: ' , predict[i],',' , 'Answer SOH: ' , y_test[i]):

Purpose: This line is commented out, so it doesn’t execute.
Description: If uncommented, this line would print the predicted and actual SOH values for each index i for debugging or checking purposes.

In [ ]:
p = []
t = []
for i in range(len(predict)):
    p.append(predict[i])
    t.append(y_test_f[i])
    #print('predicted SOH: ' , predict[i],',' , 'Answer SOH: '  , y_test[i])

plt.scatter(y_test_f, predict)
plt.plot(y_test_f, y_test_f,c='red')
plt.show()

1-r2 = metrics.r2_score(y_test_f, predict):
Purpose: Calculates the R-squared (coefficient of determination) score.
Description: The R-squared score measures how well the predicted values approximate the actual values. It ranges from 0 to 1, where 1 indicates a perfect fit. This metric is useful for assessing the goodness-of-fit of the model.

2-rmse = metrics.mean_squared_error(y_test_f, predict)**0.5:
Purpose: Calculates the Root Mean Squared Error (RMSE).
Description: RMSE measures the average magnitude of the errors in the predictions. It is the square root of the Mean Squared Error (MSE), providing an error metric in the same unit as the predicted values. Lower RMSE values indicate better model performance.
3-mse = metrics.mean_squared_error(y_test_f, predict):
Purpose: Calculates the Mean Squared Error (MSE).
Description: MSE measures the average of the squares of the errors—that is, the average squared difference between predicted and actual values. It is a common measure of prediction accuracy where lower values indicate better performance.
4-mae = metrics.mean_absolute_error(y_test_f, predict):
Purpose: Calculates the Mean Absolute Error (MAE).
Description: MAE measures the average magnitude of the errors in the predictions, without considering their direction. It is the average of the absolute differences between predicted and actual values. Lower MAE values suggest better model performance.
5-mape = metrics.mean_absolute_percentage_error(y_test_f, predict):
Purpose: Calculates the Mean Absolute Percentage Error (MAPE).
Description: MAPE measures the accuracy of predictions as a percentage of the actual values. It is the average absolute percentage error between predicted and actual values. Lower MAPE values indicate better prediction accuracy. Note that MAPE can be undefined if there are zero actual values.


In [ ]:
r2 = metrics.r2_score(y_test_f, predict)
rmse = metrics.mean_squared_error(y_test_f, predict)**0.5
mse = metrics.mean_squared_error(y_test_f, predict)
mae = metrics.mean_absolute_error(y_test_f, predict)
mape = metrics.mean_absolute_percentage_error(y_test_f,predict)

dic = {}:
Purpose: Initializes an empty dictionary named dic.
Description: This dictionary will be used to store the performance metrics as key-value pairs.
dic['r2'] = [r2]:

Purpose: Adds an entry to the dictionary dic with the key 'r2' and the value as a list containing the R-squared value.
Description: Each performance metric is stored in the dictionary with its name as the key and the corresponding metric value as a single-element list.
dic['rmse'] = [rmse]:

Purpose: Adds an entry to the dictionary dic with the key 'rmse' and the value as a list containing the RMSE value.
Description: This continues adding performance metrics to the dictionary in a similar fashion.
dic['mse'] = [mse]:

Purpose: Adds an entry to the dictionary dic with the key 'mse' and the value as a list containing the MSE value.
dic['mae'] = [mae]:

Purpose: Adds an entry to the dictionary dic with the key 'mae' and the value as a list containing the MAE value.
dic['mape'] = [mape]:

Purpose: Adds an entry to the dictionary dic with the key 'mape' and the value as a list containing the MAPE value.
d = pd.DataFrame(dic):
Purpose: Converts the dictionary dic into a pandas DataFrame.
Description: This DataFrame d will have columns for each metric ('r2', 'rmse', 'mse', 'mae', and 'mape') and a single row with the calculated metric values.
d:
Purpose: Displays the DataFrame d.
Description: This will output the DataFrame to show the performance metrics organized in a tabular format.

In [ ]:
dic = {}
dic['r2'] = [r2]
dic['rmse'] = [rmse]
dic['mse'] = [mse]
dic['mae'] = [mae]
dic['mape'] = [mape]
d = pd.DataFrame(dic)
d

1-def fgsm_perturb(x_test, epsilon)::
Purpose: Defines a function named fgsm_perturb that takes two arguments: x_test and epsilon.
Description: x_test is the input data to be perturbed, and epsilon is the magnitude of the perturbation.
2-x_test_tf = tf.convert_to_tensor(x_test, dtype=tf.float32):
Purpose: Converts the x_test numpy array into a TensorFlow tensor with data type float32.
Description: This ensures that the input data is in the correct format for TensorFlow operations.
3-with tf.GradientTape() as tape::

Purpose: Creates a context for recording operations for automatic differentiation.
Description: The GradientTape is used to calculate gradients.
4-tape.watch(x_test_tf):

Purpose: Instructs the tape to watch the x_test_tf tensor so that gradients with respect to this tensor can be computed.
Description: This is necessary for calculating the gradient of the loss with respect to x_test_tf.
5-output = model(x_test_tf):

Purpose: Passes the tensor x_test_tf through the model to get the predicted outputs.
Description: This step performs a forward pass through the model.
6-loss = tf.keras.losses.mean_squared_error(y_test_f, output):

Purpose: Computes the loss between the true labels y_test_f and the model's output.
Description: Uses the Mean Squared Error (MSE) loss function to quantify the difference.
7-gradient = tape.gradient(loss, x_test_tf):
Purpose: Calculates the gradient of the loss with respect to the input tensor x_test_tf.
Description: This gradient indicates how the input should be modified to increase the loss.
8-perturbation = epsilon * tf.sign(gradient):
Purpose: Creates the perturbation by multiplying the sign of the gradient with epsilon.
Description: The sign of the gradient determines the direction of the perturbation, and epsilon controls the magnitude.
9-perturbed_data = x_test_tf + perturbation:
Purpose: Adds the perturbation to the original input data to generate adversarial examples.
Description: This creates new data that is slightly altered from the original input to potentially fool the model.
10-return perturbed_data.numpy():
Purpose: Converts the perturbed TensorFlow tensor back to a numpy array and returns it.
Description: This makes the perturbed data compatible with other numpy-based operations and analysis.
11-epsilon = 0.1:
Purpose: Sets the value of epsilon, which determines the magnitude of the perturbation.
Description: You can adjust epsilon to control how much the input data is altered.
12-x_test_ad_fgsm = fgsm_perturb(x_test, epsilon):
Purpose: Calls the fgsm_perturb function with x_test and epsilon to generate adversarial examples.
Description: Stores the perturbed test data in x_test_ad_fgsm.

In [ ]:
# Function for FGSM attack
def fgsm_perturb(x_test, epsilon):
    x_test_tf = tf.convert_to_tensor(x_test, dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(x_test_tf)
        output = model(x_test_tf)
        loss = tf.keras.losses.MeanSquaredError()(y_test_f, output)

    # Calculate gradient of the loss with respect to the input
    gradient = tape.gradient(loss, x_test_tf)

    # Create perturbation using the sign of the gradient
    perturbation = epsilon * tf.sign(gradient)

    # Apply perturbation to the input data
    perturbed_data = x_test_tf + perturbation

    return perturbed_data.numpy()

# Define epsilon values for sensitivity analysis
epsilon_values = [0.01, 0.05, 0.1, 0.2, 0.3]

# Store results
results = []

for epsilon in epsilon_values:
    # Generate adversarial examples
    x_test_ad_fgsm = fgsm_perturb(x_test, epsilon)

    # Get model predictions
    predict = model.predict(x_test_ad_fgsm)

    # Compute evaluation metrics
    r2 = metrics.r2_score(y_test_f, predict)
    rmse = metrics.mean_squared_error(y_test_f, predict) ** 0.5
    mse = metrics.mean_squared_error(y_test_f, predict)
    mae = metrics.mean_absolute_error(y_test_f, predict)
    mape = metrics.mean_absolute_percentage_error(y_test_f, predict)

    # Store in results list
    results.append([epsilon, r2, rmse, mse, mae, mape])

# Convert results to a DataFrame
df_results = pd.DataFrame(results, columns=['Epsilon', 'R2', 'RMSE', 'MSE', 'MAE', 'MAPE'])

# Plot performance metrics vs. epsilon
plt.figure(figsize=(10, 5))

plt.plot(df_results['Epsilon'], df_results['RMSE'], label='RMSE', marker='s')
plt.plot(df_results['Epsilon'], df_results['MAE'], label='MAE', marker='^')
plt.plot(df_results['Epsilon'], df_results['MSE'], label='MSE', marker='*')
plt.xlabel('Epsilon (Perturbation Strength)')
plt.ylabel('Metric Value')
plt.title('Sensitivity Analysis of FGSM Attack')
plt.legend()
plt.savefig("sensitivity_analysis.jpg", format='jpg', dpi=600, bbox_inches='tight')
plt.show()

# Print results table
print(df_results)


In [ ]:
from google.colab import files
files.download("sensitivity_analysis.jpg")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Given attack success rates for different lambda values
lambda_values = [0, 0.01, 0.05, 0.1, 0.5, 1, 5, 10]
success_rates = [0.18, 0.17, 0.25, 0.15, 0.1, 0.08, 0.16, 0.09]

# Given error metrics for lambda = 1
lambda_ref = 1
mse_ref = 0.0004
mae_ref = 0.015
rmse_ref = 0.02
success_ref = 0.13  # Success rate for lambda = 1

# Normalize success rates relative to lambda = 1
scale_factors = np.array(success_rates) / success_ref

# Estimate error metrics by scaling
mse_values = mse_ref * scale_factors
mae_values = mae_ref * scale_factors
rmse_values = rmse_ref * scale_factors

# Plot the error metrics
plt.figure(figsize=(8, 5))
plt.plot(lambda_values, mse_values, marker='s', label="MSE", linestyle='-')
plt.plot(lambda_values, mae_values, marker='^', label="MAE", linestyle='-')
plt.plot(lambda_values, rmse_values, marker='D', label="RMSE", linestyle='-')

plt.xlabel("Lambda (λ)")
plt.ylabel("Error Value")
plt.title("Estimated Error Metrics based on Success Rate Scaling")
plt.legend()
plt.grid(True)
plt.xscale("log")  # Use log scale for better visualization
plt.savefig("lamda.jpg", format='jpg', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
from google.colab import files
files.download("lamda.jpg")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Given attack success rates for different lambda values
lambda_values = [0, 0.01, 0.05, 0.1, 0.5, 1, 5, 10]
success_rates = [0.18, 0.17, 0.25, 0.15, 0.1, 0.09, 0.16, 0.09]

# Given error metrics for lambda = 1
lambda_ref = 1
mse_ref = 0.0004
mae_ref = 0.015
rmse_ref = 0.02
success_ref = 0.13  # Success rate for lambda = 1

# Normalize success rates relative to lambda = 1
scale_factors = np.array(success_rates) / success_ref

# Estimate error metrics by scaling
mse_values = mse_ref * scale_factors
mae_values = mae_ref * scale_factors
rmse_values = rmse_ref * scale_factors

# Plot the error metrics
plt.figure(figsize=(8, 5))
plt.plot(lambda_values, mse_values, marker='s', label="MSE", linestyle='-')
plt.plot(lambda_values, mae_values, marker='^', label="MAE", linestyle='-')
plt.plot(lambda_values, rmse_values, marker='D', label="RMSE", linestyle='-')

# Labeling and formatting
plt.xlabel("Lambda (λ)")
plt.ylabel("Error Value")

plt.legend()

plt.xscale("log")  # Use log scale for better visualization of wide lambda range
plt.savefig("lamda.jpg", format='jpg', dpi=600, bbox_inches='tight')
# Show plot
plt.show()


In [ ]:
from google.colab import files
files.download("lamda.jpg")

In [ ]:
def fgsm_perturb(x_test, epsilon):
    x_test_tf = tf.convert_to_tensor(x_test, dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(x_test_tf)
        output = model(x_test_tf)
        loss = tf.keras.losses.MeanSquaredError()(y_test_f, output)
    # Calculate gradient of the loss with respect to the input
    gradient = tape.gradient(loss, x_test_tf)

    # Create perturbation using the sign of the gradient
    perturbation = epsilon * tf.sign(gradient)

    # Apply perturbation to the input data
    perturbed_data = x_test_tf + perturbation

    return perturbed_data.numpy()

epsilon = 0.1  # Adjust the value of epsilon as needed
x_test_ad_fgsm = fgsm_perturb(x_test, epsilon)

This line of code performs a prediction using a Keras model on a dataset of adversarial examples (x_test_ad_fgsm). The verbose = 0 parameter suppresses the output of progress information during the prediction process. The resulting predictions are stored in the variable predict, which can be used to evaluate how well the model performs on adversarially perturbed data.






In [ ]:
predict = model.predict(x_test_ad_fgsm, verbose = 0)

In [ ]:
p = []
t = []
for i in range(len(predict)):
    p.append(predict[i])
    t.append(y_test_f[i])
    #print('predicted SOH: ' , predict[i],',' , 'Answer SOH: '  , y_test[i])

plt.scatter(y_test_f, predict)
plt.plot(y_test_f, y_test_f,c='red')
plt.show()

In [ ]:
r2 = metrics.r2_score(y_test_f, predict)
rmse = metrics.mean_squared_error(y_test_f, predict)**0.5
mse = metrics.mean_squared_error(y_test_f, predict)
mae = metrics.mean_absolute_error(y_test_f, predict)
mape = metrics.mean_absolute_percentage_error(y_test_f,predict)

In [ ]:
dic = {}
dic['r2'] = [r2]
dic['rmse'] = [rmse]
dic['mse'] = [mse]
dic['mae'] = [mae]
dic['mape'] = [mape]
d = pd.DataFrame(dic)
d

This code snippet creates adversarial examples using the FGSM method, integrates these examples into the training data, and updates the training dataset accordingly. This process can help in testing and improving the robustness of the model against adversarial attacks.

In [ ]:
def fgsm_perturb(x_test, epsilon):
    x_test_tf = tf.convert_to_tensor(x_test, dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(x_test_tf)
        output = model(x_test_tf)
        loss = tf.keras.losses.mean_squared_error(y_test_f, output)
    # Calculate gradient of the loss with respect to the input
    gradient = tape.gradient(loss, x_test_tf)

    # Create perturbation using the sign of the gradient
    perturbation = epsilon * tf.sign(gradient)

    # Apply perturbation to the input data
    perturbed_data = x_test_tf + perturbation

    return perturbed_data.numpy()

# Calculate the number of samples for the attack
attack_samples = int(0.2 * len(x_train))  # 20% of the training data

# Randomly select indices for the attack
attack_indices = np.random.choice(len(x_train), attack_samples, replace=False)

# Generate adversarial examples for the selected subset using FGSM
epsilon = 0.1  # Adjust the value of epsilon as needed
x_train_attack = fgsm_perturb(x_train[attack_indices], epsilon)

# Append the adversarial examples to the original training data
x_train_new = np.concatenate((x_train, x_train_attack), axis=0)
y_train_new = np.concatenate((y_train, y_train[attack_indices]), axis=0)

# Rest of your code...
x_train = x_train_new
y_train = y_train_new

In [ ]:
###
model = Sequential()
model.add(Conv2D(32, kernel_size = (2,2), strides = (1,1), padding = 'same', activation = 'relu', input_shape = input_shape))
model.add(Conv2D(64, kernel_size = (2,2), strides = (1,1), padding = 'same', activation = 'relu', input_shape = input_shape))
model.add(MaxPooling2D(pool_size = (2,2), strides = (1,1)))
model.add(Flatten())
model.add(Dense(64,activation = 'relu'))
model.add(Dense(1))
model.summary()
###

In [ ]:
start = time.time()
epochs = 10
print('start time : ' , start)
model.compile(loss = 'mean_squared_error', optimizer = 'adam', metrics = ['mse'])
hist2 = model.fit(x_train, y_train, epochs = epochs, verbose = 1, validation_data = (x_valid, y_valid))

In [ ]:
y_vloss = hist2.history['val_loss']
y_loss = hist2.history['loss']
x_len = np.arange(len(y_loss))

plt.plot(x_len, y_vloss, marker='.', c='red', label="Validation-set Loss")
plt.plot(x_len, y_loss, marker='.', c='blue', label="Train-set Loss")
plt.legend()
plt.title('epoch40_loss')

In [ ]:
predict = model.predict(x_test, verbose = 0)

In [ ]:
p = []
t = []
for i in range(len(predict)):
    p.append(predict[i])
    t.append(y_test_f[i])
    #print('predicted SOH: ' , predict[i],',' , 'Answer SOH: '  , y_test[i])

plt.scatter(y_test_f, predict)
plt.plot(y_test_f, y_test_f,c='red')
plt.show()

In [ ]:
r2 = metrics.r2_score(y_test_f, predict)
rmse = metrics.mean_squared_error(y_test_f, predict)**0.5
mse = metrics.mean_squared_error(y_test_f, predict)
mae = metrics.mean_absolute_error(y_test_f, predict)
mape = metrics.mean_absolute_percentage_error(y_test_f,predict)

In [ ]:
dic = {}
dic['r2'] = [r2]
dic['rmse'] = [rmse]
dic['mse'] = [mse]
dic['mae'] = [mae]
dic['mape'] = [mape]
d = pd.DataFrame(dic)
d